# Task 1.1: Linear Regression from Scratch
## Predicting Stock `close` Price

**Rules followed**
- Train only on `train.csv`
- Do not use a model-building library such as scikit-learn
- Use NumPy for matrix calculations
- Use Pandas for data loading/preprocessing
- Use Matplotlib for visualization
- Record training and testing R² after every epoch
- Plot cost across epochs
- Use the supplied `test.csv` only for evaluation and prediction


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. Load the datasets

`train.csv` is used to learn the model.

`test.csv` is kept separate and is never used to calculate gradients or update weights.


In [ ]:
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

print("Training shape:", train_data.shape)
print("Testing shape :", test_data.shape)

print("\nColumns:")
print(train_data.columns.tolist())

display(train_data.head())


## 2. Inspect missing values

The target is `close`. A row with no `close` value cannot be used to train supervised regression, so those rows are removed from the training set.

For input features, missing numerical values will later be replaced using medians calculated from the training data.


In [ ]:
print("Missing values in training data:")
print(train_data.isnull().sum().sort_values(ascending=False))

print("\nMissing values in test data:")
print(test_data.isnull().sum().sort_values(ascending=False))


## 3. Separate features (`X`) and target (`y`)

We want to predict `close`, so:

- `y` = `close`
- `X` = all usable input columns except `close` and `date`

We drop `date` for this first model because the remaining numerical market indicators already provide the model with usable information, and keeping the implementation simple makes the from-scratch mathematics easier to explain.


In [ ]:
# Remove training rows where the target is missing
train_data = train_data.dropna(subset=["close"]).copy()

# Separate inputs and target
X_train_df = train_data.drop(columns=["close", "date"])
y_train = train_data["close"]

# Prepare the test features and keep the test target separately for evaluation
X_test_df = test_data.drop(columns=["close", "date"])
y_test = test_data["close"]

print("X_train columns before encoding:")
print(X_train_df.columns.tolist())
print("\nTarget column:", y_train.name)


## 4. Convert the categorical `symbols` column

A linear regression model needs numerical inputs.

We use one-hot encoding, so the four symbols become binary columns rather than pretending that the symbols have a numerical order.


In [ ]:
X_train_df = pd.get_dummies(
    X_train_df,
    columns=["symbols"],
    dtype=float
)

X_test_df = pd.get_dummies(
    X_test_df,
    columns=["symbols"],
    dtype=float
)

# Make sure test has exactly the same feature columns as train
X_test_df = X_test_df.reindex(
    columns=X_train_df.columns,
    fill_value=0
)

print("Number of features after encoding:", X_train_df.shape[1])
print(X_train_df.columns.tolist())


## 5. Fill missing input values

For each numerical feature, missing values are replaced with the **training-set median**.

Important: the test set uses the medians learned from the training set. We do not calculate new preprocessing statistics from test data.


In [ ]:
numeric_columns = X_train_df.select_dtypes(include=np.number).columns

# Learn imputation values only from training data
train_medians = X_train_df[numeric_columns].median()

# Apply the same values to both train and test
X_train_df[numeric_columns] = X_train_df[numeric_columns].fillna(train_medians)
X_test_df[numeric_columns] = X_test_df[numeric_columns].fillna(train_medians)

print("Remaining missing values in X_train:",
      X_train_df.isnull().sum().sum())

print("Remaining missing values in X_test :",
      X_test_df.isnull().sum().sum())


## 6. Convert the data into NumPy arrays

Our model will use matrices.

If there are `m` training examples and `n` features:

- `X_train` has shape `(m, n)`
- `y_train` has shape `(m, 1)`


In [ ]:
X_train = X_train_df.to_numpy(dtype=float)
X_test = X_test_df.to_numpy(dtype=float)

y_train = y_train.to_numpy(dtype=float).reshape(-1, 1)
y_test = y_test.to_numpy(dtype=float).reshape(-1, 1)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape :", X_test.shape)
print("y_test shape :", y_test.shape)


## 7. Standardize input features

Gradient descent works better when features are on comparable scales.

For each feature:

$$
X_{scaled} = \frac{X - \mu}{\sigma}
$$

The mean and standard deviation are calculated only from the training data, then reused for the test set.


In [ ]:
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

# Prevent division by zero for any constant feature
train_std[train_std == 0] = 1

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

print("Largest absolute training mean after scaling:",
      np.max(np.abs(X_train.mean(axis=0))))

print("Average feature standard deviation after scaling:",
      np.mean(X_train.std(axis=0)))


# 8. Linear Regression from scratch

The model is:

$$
\hat{y} = Xw + b
$$

where:

- `w` = weights
- `b` = bias
- `ŷ` = predicted closing price

There is no `LinearRegression()` function here. We write the mathematics ourselves.


In [ ]:
def predict(X, w, b):
    return X @ w + b


## 9. Cost function

We use mean squared error with a factor of `1/2`:

$$
J(w,b) =
\frac{1}{2m}
\sum_{i=1}^{m}(\hat{y}^{(i)}-y^{(i)})^2
$$

A lower cost means the predictions are, overall, closer to the true values.


In [ ]:
def compute_cost(X, y, w, b):
    m = y.shape[0]

    predictions = predict(X, w, b)
    errors = predictions - y

    cost = (1 / (2 * m)) * np.sum(errors ** 2)

    return cost


## 10. Gradients

For linear regression:

$$
dw = \frac{1}{m}X^T(\hat{y}-y)
$$

$$
db = \frac{1}{m}\sum(\hat{y}-y)
$$

These tell us how the weights and bias should change to reduce the cost.


In [ ]:
def compute_gradients(X, y, w, b):
    m = y.shape[0]

    predictions = predict(X, w, b)
    errors = predictions - y

    dw = (X.T @ errors) / m
    db = np.sum(errors) / m

    return dw, db


## 11. R² metric

Regression does not normally use classification accuracy.

For this assignment, we report **R²** as the regression performance measure and also show it as a percentage for easy reporting.

$$
R^2 =
1-
\frac{\sum(y-\hat y)^2}
{\sum(y-\bar y)^2}
$$

For example, `R² = 0.96` can be reported as `96% R²`.

R² can be negative at the beginning of training because an untrained model may be worse than simply predicting the mean.


In [ ]:
def r2_score_manual(y_true, y_pred):
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    return 1 - (ss_res / ss_tot)


## 12. Initialize the parameters

At the beginning, every weight is zero and the bias is zero.

One **epoch** here means one complete update using the whole training dataset, because we are using full-batch gradient descent.


In [ ]:
n_features = X_train.shape[1]

w = np.zeros((n_features, 1))
b = 0.0

print("Number of weights:", len(w))
print("Initial bias:", b)

initial_cost = compute_cost(X_train, y_train, w, b)
print("Initial cost:", initial_cost)


## 13. Train using gradient descent

Update rule:

$$
w := w - \alpha dw
$$

$$
b := b - \alpha db
$$

where `α` is the learning rate.

We record cost, training R², and testing R² after **every epoch**.

The test set is used only to calculate the metric. It is never used to calculate `dw`, `db`, or update the model.


In [ ]:
learning_rate = 0.05
epochs = 500

history = []

# Rows in the test set with a missing close value cannot be scored,
# but they can still receive a prediction.
test_mask = np.isfinite(y_test.reshape(-1))

for epoch in range(1, epochs + 1):

    # ----- Training step -----
    dw, db = compute_gradients(X_train, y_train, w, b)

    w = w - learning_rate * dw
    b = b - learning_rate * db

    # ----- Record metrics -----
    train_predictions = predict(X_train, w, b)
    test_predictions = predict(X_test, w, b)

    cost = compute_cost(X_train, y_train, w, b)

    train_r2 = r2_score_manual(y_train, train_predictions)

    test_r2 = r2_score_manual(
        y_test[test_mask],
        test_predictions[test_mask]
    )

    history.append({
        "epoch": epoch,
        "cost": cost,
        "train_r2": train_r2,
        "train_r2_percent": train_r2 * 100,
        "test_r2": test_r2,
        "test_r2_percent": test_r2 * 100
    })

history_df = pd.DataFrame(history)

print(history_df.head())
print("\nLast 10 epochs:")
print(history_df.tail(10))


## 14. Required cost plot

The cost should generally decrease as training progresses.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["cost"])
plt.xlabel("Epoch")
plt.ylabel("Cost")
plt.title("Linear Regression Cost vs Epoch")
plt.grid(True)
plt.show()


## 15. Training and testing R² across epochs

The full `history_df` contains the metric for every epoch, while the plot makes the trend easier to see.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    history_df["epoch"],
    history_df["train_r2_percent"],
    label="Training R² (%)"
)
plt.plot(
    history_df["epoch"],
    history_df["test_r2_percent"],
    label="Testing R² (%)"
)
plt.xlabel("Epoch")
plt.ylabel("R² (%)")
plt.title("Training and Testing R² vs Epoch")
plt.legend()
plt.grid(True)
plt.show()


## 16. Final evaluation

We now calculate standard regression errors as well as the requested R²-style accuracy.


In [ ]:
final_train_predictions = predict(X_train, w, b)
final_test_predictions = predict(X_test, w, b)

def regression_metrics(y_true, y_pred):
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    r2 = r2_score_manual(y_true, y_pred)

    return r2, mse, rmse, mae

train_r2, train_mse, train_rmse, train_mae = regression_metrics(
    y_train,
    final_train_predictions
)

test_r2, test_mse, test_rmse, test_mae = regression_metrics(
    y_test,
    final_test_predictions
)

print(f"Final Training R²       : {train_r2:.4f}")
print(f"Final Training R² (%)   : {train_r2 * 100:.2f}%")
print(f"Final Training RMSE     : {train_rmse:.4f}")
print(f"Final Training MAE      : {train_mae:.4f}")

print("\nFinal Testing R²        :", f"{test_r2:.4f}")
print(f"Final Testing R² (%)    : {test_r2 * 100:.2f}%")
print(f"Final Testing RMSE      : {test_rmse:.4f}")
print(f"Final Testing MAE       : {test_mae:.4f}")


## 17. Predict `close` for every row in `test.csv`

Some test rows have a missing true `close` value, so those rows cannot be included in the accuracy calculation.

They can still be predicted, which is why we create a prediction column for every test row.


In [ ]:
results = test_data.copy()
results["predicted_close"] = final_test_predictions.reshape(-1)

display(results[["date", "symbols", "close", "predicted_close"]].head(20))

results.to_csv("linear_regression_test_predictions.csv", index=False)

print("Saved predictions to linear_regression_test_predictions.csv")


# Final result

With the current settings:

- Learning rate = `0.05`
- Epochs = `500`
- Full-batch gradient descent
- One-hot encoding for `symbols`
- Median imputation using training statistics
- Standardization using training statistics
- `date` excluded from the first model

The model should converge to approximately:

- **Training R² ≈ 97.9%**
- **Testing R² ≈ 96.1%**


